# Notebook 2: Data Preprocessing for Sentiment Analysis

## Purpose
Clean and prepare the raw KFC Reddit comments for sentiment classification.

## Pipeline
1. Load raw CSV from Notebook 1
2. Remove rows with null/blank comments
3. Convert to lowercase, strip URLs, punctuation, special characters
4. Remove a custom KFC-specific stopword list
5. Lemmatise each word using NLTK WordNetLemmatiser
6. Store cleaned tokens in `Tokenised_Comment` column
7. Generate a word cloud for exploratory analysis
8. Save preprocessed dataset

## Input
`kfc_comments_2025_2026.csv` (from Notebook 1)

## Output
- `kfc_preprocessed_sentiment.csv`
- `wordcloud_all_comments.png`

## Step 0 — Install & Import Libraries

In [ ]:
# !pip install pandas nltk wordcloud matplotlib

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Libraries loaded.")

## Step 1 — Configuration

In [ ]:
INPUT_FILE  = "kfc_comments_2025_2026.csv"
OUTPUT_FILE = "kfc_preprocessed_sentiment.csv"
WORDCLOUD_FILE = "wordcloud_all_comments.png"

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")

## Step 2 — Define Custom Stopwords

These words appear frequently in KFC Reddit discussions but carry little **sentiment** meaning.  
Removing them helps the sentiment model focus on words that actually indicate opinion.

| Category | Examples |
|---|---|
| Brand-specific | kfc, kentucky, fried, chicken |
| Contractions | dont, cant, im, ive |
| Common verbs | go, got, get, make, want |
| Time-related | time, day, year, now, today |
| Generic | thing, one, something, back |

In [ ]:
CUSTOM_STOPWORDS = set(stopwords.words("english")).union({
    # Brand-specific
    "kfc", "kentucky", "fried", "chicken",
    # Contractions / informal
    "s", "t", "m", "don", "wa", "ha", "im", "ive", "cant", "dont",
    "doesnt", "didnt", "thats", "theyre", "u", "ur", "lol", "yeah", "ok", "okay",
    # Common verbs
    "go", "got", "get", "make", "going", "will", "would", "used",
    "use", "want", "know", "think", "work", "see", "need",
    # Time-related
    "time", "day", "year", "now", "today", "always", "ago", "last",
    # Pronouns
    "they", "them", "you", "me", "my", "their", "i", "we", "he", "she",
    # Quantity / Intensity
    "much", "still", "even", "really", "actually", "every", "everything",
    # General / Neutral
    "thing", "one", "new", "something", "back", "also", "like", "just",
    # Sentiment words too generic
    "good", "better", "bad", "pretty", "nice",
    # Order-related
    "order", "ordered",
    # Miscellaneous
    "well", "people", "place", "said", "say", "way", "will", "would", "went",
})

print(f"Custom stopword list: {len(CUSTOM_STOPWORDS)} words")

## Step 3 — Load Raw Dataset

In [ ]:
df = pd.read_csv(INPUT_FILE)
print(f"Rows loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 4 — Remove Null / Blank Comments

Rows with NaN or empty strings in the Comment column would cause errors during tokenisation.

In [ ]:
before = len(df)
df = df.dropna(subset=["Comment"])
df = df[df["Comment"].str.strip().astype(bool)]
after = len(df)

print(f"Removed {before - after} null/blank rows")
print(f"Remaining: {after} rows")

## Step 5 — Text Cleaning & Tokenisation Functions

Two functions work together:
- **`clean_text()`** — lowercase, remove URLs, Reddit markup, punctuation, numbers
- **`preprocess_comment()`** — clean → split into tokens → remove stopwords → lemmatise

In [ ]:
def clean_text(text):
    """Lowercase, remove URLs, Reddit markup, punctuation, numbers, extra spaces."""
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)     # URLs
    text = re.sub(r"/[ur]/\w+", "", text)               # Reddit usernames/subreddits
    text = re.sub(r"[^a-z\s]", "", text)                # keep only letters + spaces
    text = re.sub(r"\s+", " ", text).strip()            # collapse whitespace
    return text


def preprocess_comment(text):
    """
    Full pipeline: clean → tokenise → remove stopwords → lemmatise.
    The lemmatiser reduces words to dictionary form (e.g. 'fries' → 'fry').
    """
    lemmatizer = WordNetLemmatizer()
    cleaned = clean_text(text)
    tokens = cleaned.split()
    tokens = [t for t in tokens if t not in CUSTOM_STOPWORDS and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

# Quick test
print(preprocess_comment("I really hate the KFC app, it's always broken and my order was wrong!"))

## Step 6 — Apply Preprocessing to All Comments

In [ ]:
print("Preprocessing all comments (this may take a minute)...")
df["Tokenised_Comment"] = df["Comment"].apply(preprocess_comment)

# Remove rows that became empty after preprocessing
before = len(df)
df = df[df["Tokenised_Comment"].str.strip().astype(bool)]
after = len(df)

print(f"Removed {before - after} rows that became empty")
print(f"Final rows: {after}")
df[["Comment", "Tokenised_Comment"]].head(10)

## Step 7 — Word Cloud (Exploratory Visualisation)

A quick visual check of which words dominate the cleaned dataset.

In [ ]:
all_words = " ".join(df["Tokenised_Comment"])
wordcloud = WordCloud(
    width=1200, height=600, background_color="white",
    max_words=200, colormap="viridis",
).generate(all_words)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.title("Word Cloud of All Tokenised Comments (KFC Dataset)", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.savefig(WORDCLOUD_FILE, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to: {WORDCLOUD_FILE}")

## Step 8 — Save Preprocessed Dataset

In [ ]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved to: {OUTPUT_FILE}")
print(f"Total comments: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSample tokenised comment:")
print(df["Tokenised_Comment"].iloc[0])